# GeoVue Chip Tray Detection - Colab Training

**Setup:**
1. Upload `ml_detection` folder to Google Drive
2. Runtime > Change runtime type > **GPU (T4)**
3. Run all cells

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# UPDATE THIS PATH to where you uploaded ml_detection
DRIVE_PATH = '/content/drive/MyDrive/ml_detection'

if os.path.exists(DRIVE_PATH):
    print(f'Found: {DRIVE_PATH}')
    !ls -la {DRIVE_PATH}
else:
    print(f'Path not found: {DRIVE_PATH}')
    print('Upload ml_detection folder to Google Drive first')

In [ ]:
!pip install -q ultralytics

## Train YOLO Detector

In [ ]:
# Fix dataset.yaml paths
import yaml

yaml_path = f'{DRIVE_PATH}/yolo_dataset/dataset.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = f'{DRIVE_PATH}/yolo_dataset'
cfg['train'] = 'images/train'
cfg['val'] = 'images/val'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f)

print('Updated paths:')
!cat {yaml_path}

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=f'{DRIVE_PATH}/yolo_dataset/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    project=f'{DRIVE_PATH}/runs/detect',
    name='chip_tray',
    exist_ok=True,
    patience=20,
)
print('YOLO training complete!')

In [ ]:
# Export for Pi 5
model = YOLO(f'{DRIVE_PATH}/runs/detect/chip_tray/weights/best.pt')
model.export(format='ncnn', imgsz=640)
print('Exported to NCNN')

## Train Classifier

Upload your images to Google Drive in this structure:
```
classifier_images/
├── wet/      (images of wet samples)
├── dry/      (images of dry samples)
├── bad/      (images of bad samples)
└── empty/    (images of empty samples)
```

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torchvision import transforms, models
from PIL import Image
import random
from pathlib import Path
from collections import Counter

device = torch.device('cuda')
LABELS = ['dry', 'wet', 'empty', 'bad']  # lowercase to match folder names

# Load images from class subfolders
samples, stats = [], Counter()
classifier_path = Path(f'{DRIVE_PATH}/classifier_images')

for class_folder in classifier_path.iterdir():
    if class_folder.is_dir():
        class_name = class_folder.name.lower()
        if class_name in LABELS:
            class_idx = LABELS.index(class_name)
            for img_path in class_folder.glob('*'):
                if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                    samples.append((img_path, class_name, class_idx))
                    stats[class_name] += 1

print(f'Found {len(samples)} samples:', dict(stats))
print(f'Classes: {LABELS}')

In [ ]:
if len(samples) > 0:
    random.seed(42)
    random.shuffle(samples)
    n = len(samples)
    train_s = samples[:int(.7*n)]
    val_s = samples[int(.7*n):int(.9*n)]
    test_s = samples[int(.9*n):]

    print(f'Train: {len(train_s)}, Val: {len(val_s)}, Test: {len(test_s)}')

    class DS(Dataset):
        def __init__(s, d, t): s.d, s.t = d, t
        def __len__(s): return len(s.d)
        def __getitem__(s, i):
            img = Image.open(s.d[i][0]).convert('RGB')
            return s.t(img), s.d[i][2]

    train_t = transforms.Compose([transforms.Resize((256,256)), transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.ColorJitter(.2,.2,.2), transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    eval_t = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
        transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

    # Smaller batch size for small datasets
    batch_size = min(16, len(train_s) // 4) if len(train_s) > 16 else 4
    print(f'Using batch size: {batch_size}')

    train_dl = DataLoader(DS(train_s, train_t), batch_size=batch_size, shuffle=True, num_workers=2)
    val_dl = DataLoader(DS(val_s, eval_t), batch_size=batch_size, num_workers=2)

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier = nn.Sequential(nn.Dropout(.5), nn.Linear(576,256), nn.ReLU(), nn.Dropout(.25), nn.Linear(256,4))
    model = model.to(device)

    # Class weights for imbalanced data
    wts = torch.tensor([sum(stats.values())/(4*max(stats[l],1)) for l in LABELS]).to(device)
    print(f'Class weights: {dict(zip(LABELS, wts.tolist()))}')
    criterion = nn.CrossEntropyLoss(weight=wts)
    opt = Adam(model.parameters(), lr=.001)

    best = 0
    patience_counter = 0
    max_patience = 15  # Early stopping

    for ep in range(1, 51):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()

        model.eval()
        c, t = 0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                c += (model(x).argmax(1) == y).sum().item()
                t += y.size(0)
        acc = 100*c/t if t > 0 else 0

        if acc > best:
            best = acc
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'model_name': 'mobilenet_v3_small',
                'num_classes': 4,
                'labels': LABELS
            }, f'{DRIVE_PATH}/classifier_best.pt')
            print(f'Ep {ep}: {acc:.1f}% *')
        else:
            patience_counter += 1
            if ep % 10 == 0:
                print(f'Ep {ep}: {acc:.1f}%')

        if patience_counter >= max_patience:
            print(f'Early stopping at epoch {ep}')
            break

    print(f'Best validation accuracy: {best:.1f}%')
else:
    print('No samples found! Check your folder structure.')

## Done!

Models saved to Google Drive:
- `runs/detect/chip_tray/weights/best.pt` - YOLO
- `runs/detect/chip_tray/weights/best_ncnn_model/` - YOLO for Pi 5
- `classifier_best.pt` - Classifier

Sync Drive to your PC, then copy to Pi 5.